<a href="https://colab.research.google.com/github/prachikushwaha007-ai/Python_Pandas_RegEx_NLP/blob/main/spacy_regex_hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [174]:
import re

In [175]:
text = "Paul Newman was an American actor, but Paul Hollywood is a British TV Host. The name Paul is quite common."

In [176]:
pattern = r"Paul [A-Z]\w+"

In [177]:
matches = re.finditer(pattern, text)
for match in matches:
    print(match)

<re.Match object; span=(0, 11), match='Paul Newman'>
<re.Match object; span=(39, 53), match='Paul Hollywood'>


In [178]:
# Hybrid model of regex and spacy.

In [179]:
# The Span class represents a slice of a spaCy Doc object (a sequence of tokens). This is used later to manually construct our entity spans
import spacy
from spacy.tokens import Span

# Create a blank english language model pipeline of spacy
nlp = spacy.blank("en")
# create doc object of text
doc = nlp(text)
# convert any existing entities inside the document to a list so we can append to them later
original_ents = list(doc.ents)

# Make empty list of multi word token entity
mwt_ents = []
for match in re.finditer(pattern, doc.text):
    start, end = match.span()

# Because regex operates on character indices, we #must map those exact character spans back to spaCy's #token indices using doc.char_span(start, end).

    matched_span = doc.char_span(start, end)
#If the regex match lines up perfectly with token boundaries, m#atched_span returns a spaCy Span object. We save its #starting token index, ending token index, and text.
    if matched_span is not None:
         mwt_ents.append((matched_span.start, matched_span.end, matched_span.text))

for ent in mwt_ents:
    start, end, name = ent
    per_ent = Span(doc, start, end, label="PERSON")
    original_ents.append(per_ent)
    doc.ents = original_ents
for ent in doc.ents:
    print(f"Entity: {ent.text} | Label: {ent.label_}")

Entity: Paul Newman | Label: PERSON
Entity: Paul Hollywood | Label: PERSON


In [180]:
print(mwt_ents)

[(0, 2, 'Paul Newman'), (8, 10, 'Paul Hollywood')]


In [181]:

from spacy.language import Language
from spacy.tokens import Span
from spacy.util import filter_spans
@Language.component("paul_ner")
def paul_ner(doc):
    pattern = r"Paul [A-Z]\w+"
    original_ents = list(doc.ents)
    # Make empty list of multi word token entity
    mwt_ents = []

    for match in re.finditer(pattern, doc.text):
        start, end = match.span()

        # Because regex operates on character indices, we must map those exact character spans back to spaCy's token indices using doc.char_span(start, end).
        span = doc.char_span(start, end)
        # If the regex match lines up perfectly with token boundaries, matched_span returns a spaCy Span object. We save its starting token index, ending token index, and text.
        if span is not None:
             mwt_ents.append((span.start, span.end, span.text))

    for ent in mwt_ents:
        start, end, name = ent
        per_ent = Span(doc, start, end, label="PERSON|NORP")
        original_ents.append(per_ent)
# To avoide overlaping spans that gives value error
    filtered = filter_spans(original_ents)
    # filter_spans filters out overlapping spans, keeping the longer ones
    doc.ents = filter_spans(original_ents)
    return (doc)

In [182]:
# To add custom model create a new blank pipeline
nlp2 = spacy.load("en_core_web_sm")
nlp2.add_pipe("paul_ner")

<function __main__.paul_ner(doc)>

In [183]:
doc2 = nlp2(text)
print (doc2.ents)
for ent in doc2.ents:
   print(f"Entity: {ent.text}|Label: {ent.label_}")

(Paul Newman, American, Paul Hollywood, British, Paul)
Entity: Paul Newman|Label: PERSON
Entity: American|Label: NORP
Entity: Paul Hollywood|Label: PERSON
Entity: British|Label: NORP
Entity: Paul|Label: PERSON
